# Session 7: Dynamic Routing: RIP, OSPF and EIGRP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-07-dynamic-routing.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-07_rip_bellman_ford.py`

In [1]:
"""Session 7 - Distance-vector routing (RIP) simulated with Bellman-Ford.

Each router only knows its neighbours. Every round, each router sends its
whole table to its neighbours and keeps any route that is shorter (hop count).
RIP: metric = hops, maximum 15, 16 = unreachable.
Run:  python session-07_rip_bellman_ford.py
"""

LINKS = {("R1", "R2"), ("R2", "R3"), ("R3", "R4"), ("R1", "R5"), ("R5", "R4")}
NETWORKS = {"R1": "10.1.0.0/24", "R2": "10.2.0.0/24", "R3": "10.3.0.0/24",
            "R4": "10.4.0.0/24", "R5": "10.5.0.0/24"}
INFINITY = 16


def neighbours(r, links):
    return [b if a == r else a for a, b in links if r in (a, b)]


def converge(links, tables=None, label=""):
    routers = sorted(NETWORKS)
    if tables is None:   # start: each router knows only its own network, 0 hops
        tables = {r: {NETWORKS[r]: (0, "connected")} for r in routers}
    for rnd in range(1, 20):
        changed = False
        snapshot = {r: dict(t) for r, t in tables.items()}      # updates sent at the same time
        for r in routers:
            for n in neighbours(r, links):
                for net, (hops, _) in snapshot[n].items():
                    new = min(hops + 1, INFINITY)
                    cur = tables[r].get(net, (INFINITY, None))
                    if new < cur[0] or (cur[1] == n and new != cur[0]):
                        tables[r][net] = (new, n)
                        changed = True
        if not changed:
            print(f"{label}converged after {rnd - 1} update round(s)")
            return tables
    return tables


def show(tables, router):
    for net, (hops, via) in sorted(tables[router].items()):
        state = "unreachable" if hops >= INFINITY else f"{hops} hop(s) via {via}"
        print(f"   {router}: {net:<13} {state}")


if __name__ == "__main__":
    t = converge(LINKS, label="Initial topology: ")
    show(t, "R1")
    print("\nLink R1-R5 fails. R1 forgets routes learned through R5, then reconverges:")
    t["R1"] = {n: v for n, v in t["R1"].items() if v[1] != "R5"}
    t["R5"] = {n: v for n, v in t["R5"].items() if v[1] != "R1"}
    t = converge(LINKS - {("R1", "R5")}, t, label="After failure: ")
    show(t, "R1")

Initial topology: converged after 2 update round(s)
   R1: 10.1.0.0/24   0 hop(s) via connected
   R1: 10.2.0.0/24   1 hop(s) via R2
   R1: 10.3.0.0/24   2 hop(s) via R2
   R1: 10.4.0.0/24   2 hop(s) via R5
   R1: 10.5.0.0/24   1 hop(s) via R5

Link R1-R5 fails. R1 forgets routes learned through R5, then reconverges:
After failure: converged after 4 update round(s)
   R1: 10.1.0.0/24   0 hop(s) via connected
   R1: 10.2.0.0/24   1 hop(s) via R2
   R1: 10.3.0.0/24   2 hop(s) via R2
   R1: 10.4.0.0/24   3 hop(s) via R2
   R1: 10.5.0.0/24   4 hop(s) via R2


## `session-07_ospf_dijkstra.py`

In [2]:
"""Session 7 - Link-state routing (OSPF) simulated with Dijkstra's algorithm.

OSPF cost = reference bandwidth / interface bandwidth (default reference 100 Mbps,
so set it higher on modern networks or every link >= 100 Mbps costs 1).
Every router has the full map (LSDB) and computes the shortest-path tree itself.
Run:  python session-07_ospf_dijkstra.py
"""
import heapq

# router A, router B, link bandwidth in Mbps
LINKS = [("R1", "R2", 1000), ("R2", "R4", 1000), ("R1", "R3", 100),
         ("R3", "R4", 100), ("R1", "R4", 100), ("R4", "R5", 10000)]


def cost(bw_mbps, ref_mbps):
    return max(1, ref_mbps // bw_mbps)


def graph(ref):
    g = {}
    for a, b, bw in LINKS:
        c = cost(bw, ref)
        g.setdefault(a, []).append((b, c))
        g.setdefault(b, []).append((a, c))
    return g


def dijkstra(g, src):
    dist, prev = {src: 0}, {}
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, u
                heapq.heappush(pq, (d + w, v))
    return dist, prev


def path(prev, src, dst):
    p = [dst]
    while p[-1] != src:
        p.append(prev[p[-1]])
    return " -> ".join(reversed(p))


if __name__ == "__main__":
    for ref in (100, 100000):
        g = graph(ref)
        print(f"\nReference bandwidth {ref} Mbps; link costs:")
        for a, b, bw in LINKS:
            print(f"   {a}-{b} {bw:>6} Mbps cost {cost(bw, ref)}")
        dist, prev = dijkstra(g, "R1")
        print("   Shortest paths from R1:")
        for r in sorted(dist):
            if r != "R1":
                print(f"     to {r}: cost {dist[r]:>4}  path {path(prev, 'R1', r)}")


Reference bandwidth 100 Mbps; link costs:
   R1-R2   1000 Mbps cost 1
   R2-R4   1000 Mbps cost 1
   R1-R3    100 Mbps cost 1
   R3-R4    100 Mbps cost 1
   R1-R4    100 Mbps cost 1
   R4-R5  10000 Mbps cost 1
   Shortest paths from R1:
     to R2: cost    1  path R1 -> R2
     to R3: cost    1  path R1 -> R3
     to R4: cost    1  path R1 -> R4
     to R5: cost    2  path R1 -> R4 -> R5

Reference bandwidth 100000 Mbps; link costs:
   R1-R2   1000 Mbps cost 100
   R2-R4   1000 Mbps cost 100
   R1-R3    100 Mbps cost 1000
   R3-R4    100 Mbps cost 1000
   R1-R4    100 Mbps cost 1000
   R4-R5  10000 Mbps cost 10
   Shortest paths from R1:
     to R2: cost  100  path R1 -> R2
     to R3: cost 1000  path R1 -> R3
     to R4: cost  200  path R1 -> R2 -> R4
     to R5: cost  210  path R1 -> R2 -> R4 -> R5
